In [1]:
cd ~/PycharmProjects/CleanSpace/momia2/

/Users/jzrolling/PycharmProjects/CleanSpace/momia2


In [2]:
import momia2 as mo
import os, glob, pandas as pd, numpy as np
import tifffile
from matplotlib import pyplot as plt
from scipy.stats import pearsonr
from IPython.display import display,clear_output
from joblib import Parallel, delayed
from skimage.registration import phase_cross_correlation
import torch
import torch.nn.functional as F

TensorFlow is not installed.
tensorflow_module is not loaded because TensorFlow is not installed.


In [23]:
sample_reference = pd.read_excel('/Volumes/ExtremePro/PMM_FTSZ_RAW/sample_reference.xlsx')
include = sample_reference[sample_reference['Include for analysis']==1].copy()

batch_dict = {}
basic_norm_dict = {}
for f in sorted(glob.glob('/Volumes/ExtremePro/PMM_FTSZ_RAW/BCG_FtsZ*/')):
    batch = f.split('/')[-2]
    if batch not in batch_dict:
        batch_dict[batch] = {}
    basic_file = f+'20260831_basic.npy'
    if os.path.isfile(basic_file):
        basic_norm_dict[batch] = basic_file
    else:
        basic_norm_dict[batch] = ''
    for seg_file in sorted(glob.glob(f+'*.pk')):
        if 'autosave__' in seg_file:
            well = seg_file.split('autosave__')[-1].split('_')[0]
        elif 'autosave' in seg_file:
            well = seg_file.split('autosave_')[-1].split('_')[0]
        else:
            well = seg_file.split('Well-')[-1].split('_')[0]
        
        if len(well)!=3:
            continue
        img_file = seg_file.replace('_seg.pk','.tif')
        if well not in batch_dict[batch]:
            batch_dict[batch][well] = [[img_file,seg_file]]
        else:
            batch_dict[batch][well] += [[img_file,seg_file]]

In [16]:
def unit_func(image_file, seg_file, basic_norm, edge=64):
    img = (tifffile.imread(image_file)/basic_norm).astype(np.uint16)
    offset_corrected = np.moveaxis(mo.external.drift_corr_global(np.moveaxis(img,0,-1), 
                                                                 1,verbose=False,upsample_factor=100,max_shift=5)[0],-1,0)[:,edge:-edge,edge:-edge]
    seg = mo.pickle_load(seg_file)['labeled_mask'][edge:-edge,edge:-edge]
    image_id = image_file.split('/')[-1]
    
    p = mo.Patch(image_id=image_id,
                 image_dict={f'C{i}':offset_corrected[i] for i in range(len(offset_corrected))},
                 ref_channel='C0')
    channels = p.channels
    n_channels = len(channels)
    p.labeled_mask = seg
    p.mask = seg>0
    p.locate_particles()
    p.find_outline(use_binary=True,
                   approximate=True,
                   tolerance=0.95,interp_distance=2,
                   sigma=1,level=0.8,update_mask=True)
    p.extract_midlines(max_iteration=1,filtered_only=True,solidity_threshold=0.95)
    p.get_morphology_stats()
    p.get_intensity_stats()
    p.filter_particles({'width_max [µm]':(0.4,3),'length [µm]':(0.5,35)})
    filtered_subset = p.regionprops[p.regionprops['$include']==1].copy()
    return filtered_subset

In [52]:
for batch, v in batch_dict.items():
    basic_norm = np.load(basic_norm_dict[batch])
    for well, well_files in v.items():
        clear_output()
        print(f'Processing {batch}, well {well}...')
        cond1 = sample_reference['Batch']==batch
        cond2 = sample_reference['Well']==well
        ref_subset = sample_reference[(cond1)&(cond2)]
        if len(ref_subset)!=1:
            continue
        fold_mic, cpd, group = ref_subset.values[0][-3:]
        dst_filename = f'/Volumes/ExtremePro/PMM_FTSZ_filtered_output/{cpd}-{batch}-{well}.pk'
        if os.path.isfile(dst_filename):
            continue
        else:
            print(dst_filename)
        data_list = [(img_file, seg_file, basic_norm) for img_file, seg_file in well_files]
        filtered_data = Parallel(n_jobs=-1)(delayed(unit_func)(*x) for x in data_list)
        """
        filtered_data = []
        for img_file, seg_file, basic_norm in data_list:
            try:
                subset = unit_func(img_file, seg_file, basic_norm)
                filtered_data.append(subset)
            except:
                print(img_file)
                continue
        """
        filtered_data = pd.concat(filtered_data)
        inshade_order=32
        inshade_cols = ['InShaDe_{}'.format(x+1) for x in range(inshade_order)]
        inshade_vals = Parallel(n_jobs=-1)(delayed(mo.external.InShaDe)(x) for x in filtered_data['$outline'].values)
        filtered_data[inshade_cols] = inshade_vals
        filtered_data = filtered_data.drop(columns=['$outline', '$midlines','$coords','$include',
                                                    '$mask','$refined_outline','$touching_edge'])
        filtered_data['Batch'] = batch
        filtered_data['Well'] = well
        filtered_data['Compound'] = cpd
        filtered_data['Fold_mic'] = fold_mic
        filtered_data['Group'] = group
        mo.pickle_save(filtered_data,dst_filename) 
        

Processing BCG_FtsZ_HT_Plate10_Batch03_20250529, well C03...
